# Scientific Evidence Retrieval — Lab Notebook

Experiments comparing retrieval approaches on the [SciFact](https://github.com/allenai/scifact) dataset from the BEIR benchmark.

**Methods explored:**
1. Dense semantic search (bi-encoder + ChromaDB)
2. Cross-encoder reranking on top of dense retrieval
3. BM25 lexical search
4. Hybrid search (BM25 + dense via Reciprocal Rank Fusion)

**Metrics:** Recall@k (did we surface ≥1 relevant doc?) and nDCG@k (are relevant docs ranked near the top?)

In [ ]:
%pip install -q --disable-pip-version-check sentence-transformers scikit-learn pandas beir chromadb rank-bm25

In [ ]:
import os
import json
import logging
import warnings

# Suppress tqdm's ipywidgets warning — must be set before any library imports tqdm
warnings.filterwarnings("ignore", message="IProgress not found")

# Suppress HuggingFace Hub auth warning and model-loading progress bars
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

# Belt-and-suspenders: also silence the loggers directly
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

# Directory for persisted artifacts (ChromaDB, caches) — used in Phase 2
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

C:\Users\yaeld\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sentences = [
    "Aspirin reduces the risk of heart attack.",
    "Taking aspirin can lower the chance of myocardial infarction.",
    "The Eiffel Tower is located in Paris.",
    "Paris is home to the Eiffel Tower.",
    "Neural networks are used for machine learning.",
    "Deep learning models rely on neural networks.",
    "Bananas are yellow fruits.",
    "A car engine requires fuel.",
]

In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2981.21it/s]


In [5]:
embeddings = model.encode(sentences, normalize_embeddings=True)

In [6]:
query = "Aspirin helps prevent heart attacks."
query_embedding = model.encode([query], normalize_embeddings=True)

In [7]:
scores = cosine_similarity(query_embedding, embeddings)[0]

In [8]:
results = pd.DataFrame({
    "sentence": sentences,
    "cosine_similarity": scores
}).sort_values("cosine_similarity", ascending=False)

In [9]:
results

,sentence,cosine_similarity
0,Aspirin reduces the risk of heart attack.,0.942438
1,Taking aspirin can lower the chance of myocard...,0.808282
4,Neural networks are used for machine learning.,0.100708
5,Deep learning models rely on neural networks.,0.082284
3,Paris is home to the Eiffel Tower.,0.051832
2,The Eiffel Tower is located in Paris.,0.042877
7,A car engine requires fuel.,0.005435
6,Bananas are yellow fruits.,0.000256


## Dataset: SciFact (BEIR)

SciFact contains 5,183 scientific abstracts and 300 test queries (scientific claims), each mapped to relevant documents via expert annotations.

In [10]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

In [11]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [12]:
dataset = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

In [13]:
data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

100%|██████████| 5183/5183 [00:00<00:00, 102847.97it/s]


In [14]:
print(type(corpus))
print(type(queries))
print(type(qrels))

<class 'dict'>
<class 'dict'>
<class 'dict'>


In [15]:
print("Corpus size:", len(corpus))
print("Queries size:", len(queries))
print("Qrels size:", len(qrels))

corpus_df = pd.DataFrame.from_dict(corpus, orient="index").reset_index()
corpus_df = corpus_df.rename(columns={"index": "_id"})

queries_df = pd.DataFrame.from_dict(queries, orient="index", columns=["text"]).reset_index()
queries_df = queries_df.rename(columns={"index": "_id"})

qrels_rows = []

for query_id, relevant_docs in qrels.items():
    for doc_id, score in relevant_docs.items():
        qrels_rows.append({"query-id": query_id, "corpus-id": doc_id, "score": score})

qrels_df = pd.DataFrame(qrels_rows)
display(corpus_df.head())
display(queries_df.head())
display(qrels_df.head())

Corpus size: 5183
Queries size: 300
Qrels size: 300


,_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...


,_id,text
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...


,query-id,corpus-id,score
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1


In [16]:
query_overlap = qrels_df["query-id"].isin(queries_df["_id"]).mean()
corpus_overlap = qrels_df["corpus-id"].isin(corpus_df["_id"]).mean()

print("Query overlap:", query_overlap)
print("Corpus overlap:", corpus_overlap)

Query overlap: 1.0
Corpus overlap: 1.0


In [17]:
for i in range(5):
    row = qrels_df.iloc[i]

    query_id = row["query-id"]
    corpus_id = row["corpus-id"]

    query_text = queries_df.loc[queries_df["_id"] == query_id, "text"].iloc[0]
    doc = corpus_df.loc[corpus_df["_id"] == corpus_id].iloc[0]

    print("=" * 100)
    print("QUERY ID:", query_id)
    print("QUERY:", query_text)
    print()
    print("RELEVANT DOC ID:", corpus_id)
    print("TITLE:", doc["title"])
    print("TEXT:", doc["text"][:1000])
    print()

QUERY ID: 1
QUERY: 0-dimensional biomaterials show inductive properties.

RELEVANT DOC ID: 31715818
TITLE: New opportunities: the use of nanotechnologies to manipulate and track stem cells.
TEXT: Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo tracking; nanoparticles, carbon nanotubes, and polyplexes for the intracellular delivery of genes/oligonucleotides and protein/peptides; and engineered nanometer-scale scaffolds for stem cell differentiation and transplantation. This review examines the use of nanotechnologies for stem cell tracking, differentiation, and transplantation. We further discuss their utility and the potential concerns regarding their cytotoxicity.

QUERY ID: 3
QUERY: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common vari

## Semantic Search (Dense Retrieval)

Encode all documents once with a bi-encoder (`all-MiniLM-L6-v2`) and store in ChromaDB. At query time, encode the query and retrieve the nearest neighbors by cosine similarity.

In [18]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5362.23it/s]


In [19]:
documents = []
metadatas = []
ids = []

for _, row in corpus_df.iterrows():
    doc_text = f"{row['title']} {row['text']}"
    documents.append(doc_text)
    metadatas.append({"title": row["title"]})
    ids.append(str(row["_id"]))

In [20]:
subset_size = 500

subset_docs = documents[:subset_size]
subset_ids = ids[:subset_size]
subset_metadata = metadatas[:subset_size]

In [21]:
doc_embeddings = embedding_model.encode(
    subset_docs,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 16/16 [00:29<00:00,  1.83s/it]


In [22]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(name="scifact")

In [23]:
collection.add(
    documents=subset_docs,
    embeddings=doc_embeddings.tolist(),
    metadatas=subset_metadata,
    ids=subset_ids
)

In [24]:
def search(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results

In [25]:
query = "Does aspirin reduce heart attack risk?"

results = search(query)

for i in range(len(results["documents"][0])):
    print("=" * 80)
    print("DOC ID:", results["ids"][0][i])
    print("TITLE:", results["metadatas"][0][i]["title"])
    print()
    print(results["documents"][0][i][:1000])

DOC ID: 1287809
TITLE: Cost-effectiveness of 10-Year Risk Thresholds for Initiation of Statin Therapy for Primary Prevention of Cardiovascular Disease.

Cost-effectiveness of 10-Year Risk Thresholds for Initiation of Statin Therapy for Primary Prevention of Cardiovascular Disease. IMPORTANCE The American College of Cardiology and the American Heart Association (ACC/AHA) cholesterol treatment guidelines have wide-scale implications for treating adults without history of atherosclerotic cardiovascular disease (ASCVD) with statins. OBJECTIVE To estimate the cost-effectiveness of various 10-year ASCVD risk thresholds that could be used in the ACC/AHA cholesterol treatment guidelines. DESIGN, SETTING, AND PARTICIPANTS Microsimulation model, including lifetime time horizon, US societal perspective, 3% discount rate for costs, and health outcomes. In the model, hypothetical individuals from a representative US population aged 40 to 75 years received statin treatment, experienced ASCVD events,

In [26]:
queries_to_test = [
    "COVID vaccine effectiveness",
    "brain cancer treatment",
    "heart disease prevention",
    "gene mutation effects",
    "protein folding"
]

for q in queries_to_test:
    print("=" * 80)
    print("QUERY:", q)

    results = search(q, top_k=3)

    for i in range(3):
        print("--- RESULT", i + 1)
        print("TITLE:", results["metadatas"][0][i]["title"])
        print(results["documents"][0][i][:500])

QUERY: COVID vaccine effectiveness
--- RESULT 1
TITLE: Vaccines against malaria
Vaccines against malaria There is no licenced vaccine against any human parasitic disease and Plasmodium falciparum malaria, a major cause of infectious mortality, presents a great challenge to vaccine developers. This has led to the assessment of a wide variety of approaches to malaria vaccine design and development, assisted by the availability of a safe challenge model for small-scale efficacy testing of vaccine candidates. Malaria vaccine development has been at the forefront of assessing ma
--- RESULT 2
TITLE: Intermittent prophylaxis with oral truvada protects macaques from rectal SHIV infection.
Intermittent prophylaxis with oral truvada protects macaques from rectal SHIV infection. HIV continues to spread globally, mainly through sexual contact. Despite advances in treatment and care, preventing transmission with vaccines or microbicides has proven difficult. A promising strategy to avoid transmissi

In [27]:
import chromadb

client = chromadb.Client()

# Delete old 500-doc collection if it exists
try:
    client.delete_collection("scifact")
except:
    pass

collection = client.create_collection(name="scifact")

In [28]:
doc_embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    normalize_embeddings=True,
    batch_size=32
)

Batches: 100%|██████████| 162/162 [06:00<00:00,  2.22s/it]


In [29]:
collection.add(
    documents=documents,
    embeddings=doc_embeddings.tolist(),
    metadatas=metadatas,
    ids=ids
)

In [30]:
for q in queries_to_test:
    print("=" * 80)
    print("QUERY:", q)

    results = search(q, top_k=3)

    for i in range(3):
        print("--- RESULT", i + 1)
        print("TITLE:", results["metadatas"][0][i]["title"])
        print(results["documents"][0][i][:500])

QUERY: COVID vaccine effectiveness
--- RESULT 1
TITLE: Pandemic H1N12009 influenza and HIV: a review of natural history, management and vaccine immunogenicity.
Pandemic H1N12009 influenza and HIV: a review of natural history, management and vaccine immunogenicity. PURPOSE OF REVIEW The 2009 pandemic HIN1 influenza strain (H1N12009) produced more severe disease and increased risk for mortality. As an at-risk population for more severe influenza illness, particular concern regarding HIV patients triggered a focused effort to evaluate disease burden and vaccine efficacy in these populations. RECENT FINDINGS As with other immune-compromised individuals, mo
--- RESULT 2
TITLE: Accelerating Policy Decisions to Adopt Haemophilus influenzae Type b Vaccine: A Global, Multivariable Analysis
Accelerating Policy Decisions to Adopt Haemophilus influenzae Type b Vaccine: A Global, Multivariable Analysis BACKGROUND Adoption of new and underutilized vaccines by national immunization programs is an ess

In [31]:
def search_ids(query, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results["ids"][0]

In [35]:
import math

def dcg_at_k(retrieved_ids, relevant_docs, k=10):
    """Discounted Cumulative Gain: rewards relevant docs ranked higher.
    score += rel / log2(rank + 1) for each relevant doc in the top-k.
    """
    score = 0.0

    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        relevance = relevant_docs.get(doc_id, 0)

        if relevance > 0:
            score += relevance / math.log2(rank + 1)

    return score

In [36]:
def evaluate_retrieval(search_fn, k=10):
    """Compute Recall@k and nDCG@k for any search function (query_text -> list[doc_id]).

    Returns (recall, ndcg) as floats.
    """
    recall_hits = 0
    ndcg_scores = []

    for query_id, relevant_docs in qrels.items():
        retrieved_ids = search_fn(queries[query_id])[:k]
        relevant_set = set(relevant_docs.keys())

        # Recall: did we retrieve at least one relevant doc?
        if set(retrieved_ids) & relevant_set:
            recall_hits += 1

        # nDCG: are relevant docs ranked highly?
        dcg = dcg_at_k(retrieved_ids, relevant_docs, k)
        ideal_relevances = sorted(relevant_docs.values(), reverse=True)
        ideal_docs = {str(i): rel for i, rel in enumerate(ideal_relevances)}
        idcg = dcg_at_k([str(i) for i in range(len(ideal_relevances))], ideal_docs, k)

        if idcg > 0:
            ndcg_scores.append(dcg / idcg)

    return recall_hits / len(qrels), sum(ndcg_scores) / len(ndcg_scores)

In [37]:
# Evaluate semantic baseline at k=5 and k=10
recall_5, _ = evaluate_retrieval(lambda q: search_ids(q, top_k=5), k=5)
recall_10, ndcg_10 = evaluate_retrieval(lambda q: search_ids(q, top_k=10), k=10)

In [38]:
print(f"Recall@5:  {recall_5:.3f}")
print(f"Recall@10: {recall_10:.3f}")
print(f"nDCG@10:   {ndcg_10:.3f}")

Recall@5:  0.753
Recall@10: 0.793
nDCG@10:   0.645


## Cross-Encoder Reranking

The two-stage pipeline: dense retrieval (ChromaDB) retrieves a broader candidate set (top-50), then a cross-encoder reranker scores each (query, doc) pair directly to produce a more precise final ranking (top-10).

The cross-encoder trades off speed for accuracy. It's slower than the bi-encoder but considers the full query-document interaction.

In [39]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2980.38it/s]


In [40]:
def search_ids_reranked(query, retrieve_k=50, final_k=10):
    # Stage 1: retrieve top-retrieve_k candidates with dense semantic search
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=retrieve_k
    )

    candidate_ids = results["ids"][0]
    candidate_docs = results["documents"][0]

    # Stage 2: rerank candidates with the cross-encoder
    pairs = [(query, doc) for doc in candidate_docs]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    ranked = sorted(
        zip(candidate_ids, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc_id for doc_id, score in ranked[:final_k]]

In [41]:
reranked_cache = {}

def build_reranked_cache(retrieve_k=50, final_k=10):
    """Populate the reranked cache for all queries.
    Run once, metric functions then read from cache instead of calling the reranker each time.
    """
    for query_id, query_text in queries.items():
        reranked_cache[query_id] = search_ids_reranked(
            query_text,
            retrieve_k=retrieve_k,
            final_k=final_k
        )

build_reranked_cache(retrieve_k=50, final_k=10)

In [56]:
def ndcg_from_cache(cache, k=10):
    scores = []

    for query_id, relevant_docs in qrels.items():
        retrieved_ids = cache[query_id][:k]

        dcg = dcg_at_k(retrieved_ids, relevant_docs, k)

        ideal_relevances = sorted(relevant_docs.values(), reverse=True)
        ideal_docs = {str(i): rel for i, rel in enumerate(ideal_relevances)}
        ideal_retrieved = [str(i) for i in range(len(ideal_relevances))]

        idcg = dcg_at_k(ideal_retrieved, ideal_docs, k)

        if idcg > 0:
            scores.append(dcg / idcg)

    return sum(scores) / len(scores)

In [ ]:
print(f"nDCG@10 before reranking: {ndcg_10:.3f}")
print(f"nDCG@10 after reranking:  {ndcg_from_cache(reranked_cache, 10):.3f}")

In [55]:
def recall_from_cache(cache, k=10):
    hits = 0

    for query_id, relevant_docs in qrels.items():
        retrieved_ids = cache[query_id][:k]
        relevant_doc_ids = set(relevant_docs.keys())

        if set(retrieved_ids) & relevant_doc_ids:
            hits += 1

    return hits / len(qrels)

In [ ]:
print(f"Recall@10 before reranking: {recall_10:.3f}")
print(f"Recall@10 after reranking:  {recall_from_cache(reranked_cache, 10):.3f}")

## BM25 Keyword Search Baseline

BM25 (Okapi BM25) is a classic lexical ranking function based on term frequency and inverse document frequency. It serves as a strong baseline and complements dense retrieval. BM25 excels at exact keyword matches, while dense retrieval handles synonyms and paraphrases.

In [46]:
from rank_bm25 import BM25Okapi

tokenized_docs = [
    doc.lower().split()
    for doc in documents
]

bm25 = BM25Okapi(tokenized_docs)

In [47]:
def bm25_search(query, top_k=10):

    tokenized_query = query.lower().split()

    scores = bm25.get_scores(tokenized_query)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    return [
        ids[i]
        for i in ranked_indices
    ]

In [48]:
recall_bm25, ndcg_bm25 = evaluate_retrieval(bm25_search, k=10)

In [49]:
print(f"BM25 Recall@10: {recall_bm25:.3f}")
print(f"BM25 nDCG@10:   {ndcg_bm25:.3f}")

BM25 Recall@10: 0.703
BM25 nDCG@10:   0.560


## Hybrid Search (Reciprocal Rank Fusion)

Combine BM25 and dense retrieval without any new dependencies using **Reciprocal Rank Fusion (RRF)**:

```
score(doc) = Σ_method  1 / (k + rank(doc, method))
```

where `k=60` is the standard smoothing constant. Each method contributes a score inversely proportional to the document's rank. Documents appearing in both lists get a boost from both contributions.

In [50]:
def hybrid_search(query, top_k=10, retrieve_k=50, rrf_k=60):
    """Combine BM25 and dense retrieval via Reciprocal Rank Fusion (RRF).

    Both methods retrieve retrieve_k candidates independently, then RRF
    merges their ranked lists into a single score. rrf_k=60 is the standard
    smoothing constant that dampens the impact of very high ranks.
    """
    # Dense retrieval candidates
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    dense_results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=retrieve_k
    )
    dense_ids = dense_results["ids"][0]

    # BM25 candidates
    bm25_ids = bm25_search(query, top_k=retrieve_k)

    # RRF fusion: accumulate scores from both ranked lists
    rrf_scores = {}
    for rank, doc_id in enumerate(dense_ids, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (rrf_k + rank)
    for rank, doc_id in enumerate(bm25_ids, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (rrf_k + rank)

    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in ranked[:top_k]]

In [51]:
recall_hybrid, ndcg_hybrid = evaluate_retrieval(hybrid_search, k=10)
print(f"Hybrid Recall@10: {recall_hybrid:.3f}")
print(f"Hybrid nDCG@10:   {ndcg_hybrid:.3f}")

Hybrid Recall@10: 0.803
Hybrid nDCG@10:   0.646


### Hybrid RRF + Cross-Encoder Reranking

Same pipeline but with a reranking step after fusion: dense + BM25 → RRF → top-50 → cross-encoder → top-10.

In [52]:
# Precompute a fast id → document text lookup (used in hybrid reranking)
_id_to_doc_text = dict(zip(ids, documents))

def hybrid_search_reranked(query, retrieve_k=100, rrf_k=60, final_k=10):
    """BM25 + dense → RRF fusion → cross-encoder rerank.

    retrieve_k is intentionally larger than in RRF-only to give the fusion
    a wider candidate pool before handing off to the reranker.
    """
    # Dense retrieval (fetch docs too — needed for reranking)
    query_embedding = embedding_model.encode([query], normalize_embeddings=True)[0]
    dense_results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=retrieve_k
    )
    dense_ids = dense_results["ids"][0]
    id_to_doc = dict(zip(dense_ids, dense_results["documents"][0]))

    # BM25 candidates
    bm25_ids = bm25_search(query, top_k=retrieve_k)

    # RRF fusion → top-50 candidates for the reranker
    rrf_scores = {}
    for rank, doc_id in enumerate(dense_ids, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (rrf_k + rank)
    for rank, doc_id in enumerate(bm25_ids, start=1):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (rrf_k + rank)

    rrf_top50 = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:50]
    candidate_ids = [doc_id for doc_id, _ in rrf_top50]

    # Fetch document text — dense results cover most; fall back to corpus for BM25-only candidates
    candidate_docs = [id_to_doc.get(doc_id, _id_to_doc_text[doc_id]) for doc_id in candidate_ids]

    # Rerank with cross-encoder
    pairs = [(query, doc) for doc in candidate_docs]
    scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    ranked = sorted(zip(candidate_ids, scores), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in ranked[:final_k]]

In [53]:
hybrid_reranked_cache = {}

def build_hybrid_reranked_cache(retrieve_k=100, rrf_k=60, final_k=10):
    """Populate the hybrid reranked cache for all queries.
    Run once — metric functions then read from cache.
    """
    for query_id, query_text in queries.items():
        hybrid_reranked_cache[query_id] = hybrid_search_reranked(
            query_text,
            retrieve_k=retrieve_k,
            rrf_k=rrf_k,
            final_k=final_k
        )

build_hybrid_reranked_cache()

In [57]:
print(f"Hybrid RRF + Reranking Recall@10: {recall_from_cache(hybrid_reranked_cache, 10):.3f}")
print(f"Hybrid RRF + Reranking nDCG@10:   {ndcg_from_cache(hybrid_reranked_cache, 10):.3f}")

Hybrid RRF + Reranking Recall@10: 0.830
Hybrid RRF + Reranking nDCG@10:   0.692


## Results Summary

| Method | Recall@10 | nDCG@10 |
|--------|-----------|---------|
| Dense (Semantic) | 0.793 | 0.642 |
| BM25 | 0.703 | 0.560 |
| Dense + Reranking | 0.833 | 0.690 |
| Hybrid RRF (no rerank) | 0.803 | 0.646 |
| Hybrid RRF + Reranking | 0.830 | 0.692 |